In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(df)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
# Plot the target distribution (delivery_time)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'], bins=50, edgecolor='black')
plt.title('Delivery time Distribution')
plt.xlabel('Delivery time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df.drop(columns=['Order_ID'], inplace= True)

In [ ]:
# Task 2: Write your code here:
df.isnull().sum()

In [ ]:
df_clean = df.copy()

# Drop rows where target (price) or key features are missing - can't predict without them
print(f"Before: {df_clean.shape}")

df_clean = df_clean.dropna(subset=['Delivery_Time']).copy()

for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna('unknown')

df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mean())

print("Missing values remaining:", df_clean.isnull().sum().sum())
print(f"After dropping missing: {df_clean.shape}")

In [ ]:
# Task 3: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder #import LabelEncoder


label_encoder = LabelEncoder()
df_clean["Traffic_Level"] = label_encoder.fit_transform(df_clean["Traffic_Level"])

onehot_encoder = OneHotEncoder(sparse_output=False)
df_clean['Time_of_Day'] = onehot_encoder.fit_transform(df_clean)
df_clean['Weather'] = onehot_encoder.fit_transform(df_clean)
df_clean['Vehicle_Type'] = onehot_encoder.fit_transform(df_clean)
df_clean


In [ ]:
# Task 5: Write your code here:
# Apply feature scaling for all features

from sklearn.preprocessing import StandardScaler
feature_cols = ['Distance_km',	'Weather',	'Traffic_Level',	'Time_of_Day',	'Vehicle_Type',	'Preparation_Time_min',	'Courier_Experience_yrs']

df_scled = df_clean[feature_cols]
standard_scaler = StandardScaler()
df_scled = standard_scaler.fit_transform(df_scled) # Apply fit_transform


In [ ]:
df_scled

In [ ]:
# Task 6: Write your code here:
# Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)





In [ ]:
# Task 1: Write your code here:
X = df_scled
y = df_clean['Delivery_Time']

In [ ]:
print("\nDataset Shapes")
print("X:", X.shape)
print("y:", y.shape)

In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

model= RandomForestRegressor()


 # K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation metrics
    mae_scores.append(mean_absolute_error(y_test, y_pred))


# Print Evaluation Metrics
print("\nModel Evaluation Metrics (K-Fold)\n" + "-"*40)
print(f"MAE : {np.mean(mae_scores):.2f}")
print("-"*40)

# Plot Predictions vs Ground Truth
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--",
    linewidth=2
)

plt.xlabel("Actual Exam Scores (Ground Truth)")
plt.ylabel("Predicted Exam Scores")
plt.title("Linear Regression: Predictions vs Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(y_pred, edgecolor='black')
plt.title('Delivery time Distribution')
plt.xlabel('Delivery time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here:

